In [103]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from pydantic import SecretStr,BaseModel , Field 
from typing import TypedDict,NotRequired,Annotated
from langgraph.graph import StateGraph, START, END
import operator

from dotenv import load_dotenv

load_dotenv()

True

In [104]:
llm = ChatGoogleGenerativeAI(
    model = "gemini-3.5-flash-lite",
    api_key = SecretStr(os.environ["GOOGLE_API_KEY"]),
)

In [105]:
class EvaluationSchema(BaseModel):
    feedback : str = Field(...,description = "Detailed feedback for the essay")
    score :int = Field(...,description = "score out of 10",ge=0,le=10)

In [106]:
structured_llm = llm.with_structured_output(EvaluationSchema)

In [107]:
essay = """The dawn of the artificial intelligence revolution has reshaped the digital landscape, elevating the **AI engineer** from a niche technical role to the architect of our modern future. These professionals are the visionary builders who translate theoretical machine learning concepts into tangible applications—from life-saving healthcare diagnostics to autonomous vehicles and intuitive conversational agents. Becoming an AI engineer is not merely about mastering a single coding language; it is a rigorous, multidisciplinary journey that demands a fusion of logical precision, mathematical fluency, and relentless curiosity.

The foundation of this career begins with a robust understanding of **computer science and mathematics**. While traditional pathways often involve degrees in computer science, data science, or engineering, the industry increasingly values demonstrable skills over formal credentials. A profound grasp of **linear algebra, calculus, probability, and statistics** is non-negotiable. These mathematical concepts are the invisible engines powering the algorithms that allow machines to learn, adapt, and predict.

Equally critical is a deep mastery of programming. **Python** reigns supreme in the AI ecosystem due to its simplicity and the vast array of libraries it supports, though proficiency in languages like C++, Java, or R is incredibly beneficial. An aspiring AI engineer must immerse themselves in data manipulation and preprocessing, recognizing that the quality of an AI model is inextricably linked to the quality of the data it consumes.

Beyond syntax and equations, the true craft of AI engineering lies in mastering **machine learning and deep learning frameworks**. Familiarity with powerful tools such as **TensorFlow, PyTorch, and Keras** allows engineers to build, train, and deploy complex neural networks and large language models. The landscape is also evolving to demand expertise in deployment strategies and cloud architecture, as a model is only as useful as its ability to scale reliably in real-world applications.

Transitioning from a learner to an employable professional requires a **tangible portfolio**. Building end-to-end projects—such as recommendation systems, natural language processors, or computer vision applications—serves as irrefutable proof of your capabilities. Coupled with indispensable soft skills like **critical thinking, adaptability, and effective communication**, an aspiring engineer can successfully navigate technical interviews and collaborate across diverse teams.

Ultimately, becoming an AI engineer is a commitment to perpetual learning. In a field that transforms overnight, the most successful engineers are those who view artificial intelligence not just as a set of tools, but as an ever-expanding frontier of human ingenuity."""

In [108]:
class UPSCState(TypedDict):

    essay : str
    language_feedback : NotRequired[str]
    analytical_feedback : NotRequired[str]
    clarity_feedback : NotRequired[str]
    overall_feedback : NotRequired[str]

    individual_score : Annotated[list[NotRequired[int]],operator.add]
    avg_score: NotRequired[float]







In [109]:
def evaluate_language(state :UPSCState) :
    prompt = f"Evaluate the language quality of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"

    output = structured_llm.invoke(prompt)

    return {
        'language_feedback': output.feedback,
        'individual_score': [output.score]  
    }

In [110]:
def evaluate_analysis(state : UPSCState) :
    prompt = f"Evaluate the analytical quality of the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"

    output = structured_llm.invoke(prompt)

    return {
        'analytical_feedback': output.feedback,
        'individual_score': [output.score]  
    }

In [111]:
def evaluate_thought(state : UPSCState) :
    prompt = f"Evaluate the clarity of thought in the following essay and provide feedback and a score out of 10:\n\n{state['essay']}"

    output = structured_llm.invoke(prompt)

    return {
        'clarity_feedback': output.feedback,
        'individual_score': [output.score]  
    }

In [112]:
def evaluate_overall(state :UPSCState):
    prompt= f"Based on the following feedbacks, create summarize feedback:\n\nLanguage Feedback: {state['language_feedback']}\nAnalytical Feedback: {state['analytical_feedback']}\nClarity Feedback: {state['clarity_feedback']}"
    overall_feedback = llm.invoke(prompt).text

    avg_scores = sum(state['individual_score'])/len(state['individual_score'])
    return {
        'overall_feedback': overall_feedback,
        'avg_score': [avg_scores]
    }





In [113]:
graph = StateGraph(UPSCState)

graph.add_node("evaluate_language",evaluate_language)
graph.add_node("evaluate_analysis",evaluate_analysis)
graph.add_node("evaluate_thought",evaluate_thought)
graph.add_node("evaluate_overall",evaluate_overall)

graph.add_edge(START,"evaluate_language")
graph.add_edge(START,"evaluate_analysis")
graph.add_edge(START,"evaluate_thought")

graph.add_edge("evaluate_language","evaluate_overall")
graph.add_edge("evaluate_analysis","evaluate_overall")
graph.add_edge("evaluate_thought","evaluate_overall")

graph.add_edge("evaluate_overall",END)

workflow = graph.compile()

final_state = workflow.invoke({"essay":essay})

final_state

{'essay': 'The dawn of the artificial intelligence revolution has reshaped the digital landscape, elevating the **AI engineer** from a niche technical role to the architect of our modern future. These professionals are the visionary builders who translate theoretical machine learning concepts into tangible applications—from life-saving healthcare diagnostics to autonomous vehicles and intuitive conversational agents. Becoming an AI engineer is not merely about mastering a single coding language; it is a rigorous, multidisciplinary journey that demands a fusion of logical precision, mathematical fluency, and relentless curiosity.\n\nThe foundation of this career begins with a robust understanding of **computer science and mathematics**. While traditional pathways often involve degrees in computer science, data science, or engineering, the industry increasingly values demonstrable skills over formal credentials. A profound grasp of **linear algebra, calculus, probability, and statistics*